# Breast Cancer Wisconsin (Diagnostic): EDA and Data Preparation

**Problem:** Binary classification of breast tumours as benign (B) or malignant (M).

**Dataset:** UCI Breast Cancer Wisconsin (Diagnostic), supplied as `breast_cancer_wisconsin_diagnostic.csv`.

This notebook performs initial data inspection, exploratory data analysis, and a stratified train-test split. It deliberately does not fit models yet. The training-only preprocessing will be placed inside Scikit-learn pipelines in the model-training stage to prevent data leakage.

**Run instructions:** Keep this notebook and the CSV in the same directory, then run all cells from top to bottom.


## 1. Import libraries and load the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
DATA_PATH = "breast_cancer_wisconsin_diagnostic.csv"

df = pd.read_csv(DATA_PATH)
df.head()


## 2. Dataset structure and data quality

In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())
print("\\nData types:")
print(df.dtypes.value_counts())
print("\\nMissing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("\\nTarget counts:")
print(df["diagnosis"].value_counts())
print("\\nTarget percentages:")
print((df["diagnosis"].value_counts(normalize=True) * 100).round(2))


## 3. Descriptive statistics

In [ ]:
X = df.drop(columns="diagnosis")
y = df["diagnosis"].map({"B": 0, "M": 1})  # B=0, M=1

display(X.describe().T)


## 4. Class distribution

In [ ]:
plt.figure(figsize=(6, 4))
df["diagnosis"].value_counts().reindex(["B", "M"]).plot(kind="bar")
plt.title("Diagnosis Class Distribution")
plt.xlabel("Diagnosis (B = Benign, M = Malignant)")
plt.ylabel("Number of observations")
plt.tight_layout()
plt.show()


## 5. Feature distributions

In [ ]:
selected_features = [
    "radius_mean", "texture_mean", "perimeter_mean",
    "area_mean", "smoothness_mean", "compactness_mean"
]

df_melt = df.melt(
    id_vars="diagnosis",
    value_vars=selected_features,
    var_name="Feature",
    value_name="Value"
)

g = sns.displot(
    data=df_melt, x="Value", hue="diagnosis", palette="deep",
    col="Feature", col_wrap=3, kind="hist",
    facet_kws={"sharex": False, "sharey": False},
    height=3, aspect=1.2
)
g.fig.suptitle("Selected Feature Distributions by Diagnosis", y=1.03)
plt.show()


## 6. Outlier inspection using boxplots

In [ ]:
plt.figure(figsize=(12, 6))
plot_features = ["radius_mean", "texture_mean", "perimeter_mean",
                 "area_mean", "smoothness_mean", "compactness_mean"]
sns.boxplot(data=df[plot_features], orient="h")
plt.title("Boxplots of Selected Features")
plt.xlabel("Feature value")
plt.tight_layout()
plt.show()

# Boxplot flags are exploratory only; do not automatically delete observations.


## 7. Feature correlation

In [ ]:
corr = X.corr(numeric_only=True)

plt.figure(figsize=(18, 14))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.1)
plt.title("Correlation Heatmap of the 30 Predictor Features")
plt.tight_layout()
plt.show()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
display(upper.stack().sort_values(ascending=False).head(15).rename("Correlation").to_frame())


## 8. Stratified train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("\\nTraining target distribution:")
print(y_train.value_counts().sort_index())
print("\\nTesting target distribution:")
print(y_test.value_counts().sort_index())


## 9. Save the held-out test data for the Streamlit application

The test CSV must contain all 30 predictor columns and the target column. The app will load saved models and evaluate this test data without retraining.

The test file is created here. Do not use this test set to fit preprocessing steps or tune model hyperparameters.


In [ ]:
test_data = X_test.copy()
test_data["diagnosis"] = y_test.map({0: "B", 1: "M"})
test_data.to_csv("test_data.csv", index=False)
print("Saved test_data.csv with shape:", test_data.shape)


## 10. Notes for the modelling stage

- `diagnosis` is the target; the 30 numerical columns are predictors.
- The target encoding used for modelling is B=0 and M=1.
- The split is stratified so the class proportions are approximately preserved.
- Fit `StandardScaler` only on training data, inside a Scikit-learn `Pipeline`.
- Scaling is generally needed for Logistic Regression, KNN and SVM; Decision Tree and Gaussian Naive Bayes do not require standardization in the same way.
- Do not remove observations solely because a boxplot marks them as potential outliers. Investigate before making such a decision.
- Use `predict_proba` or `decision_function` for ROC-AUC, as appropriate.
